# Big Data Platforms — Lecture 3 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC Exam, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 8.9.2026

This notebook is my working set of notes from lecture 3, expanded with explanations
and a couple of runnable code snippets. The lecture covers three things that all fit
together: the Spark ecosystem (libraries, RDDs, DataFrames, broadcast variables,
accumulators), a recap of how MapReduce actually works under the hood, and HDFS —
the filesystem most of this still sits on top of.

I've kept the original slide structure roughly intact so it's easy to cross-reference
against the PDF, but I've written out the reasoning in full sentences instead of
bullet fragments.


## 1. Exam logistics

A few practical points, mostly so I don't forget them:

- Register separately for the exam course **DATA143032** (2 ECTS) — Sisu for UH
  students, Open University registration for everyone else. The lecture course
  itself doesn't automatically enrol you in the exam.
- The exam is sat in person at **Examinarium**, open between **19.10.2026 and
  9.12.2026**. So there's a roughly seven-week window to book a slot — worth doing
  early rather than in the last week when everyone else is trying to do the same.
- Grading happens **10.12–18.12**, so results land before the winter break.


## 2. Reading list

Two tracks here — one for Spark, one for Hadoop/HDFS.

**Spark**
- Drabas & Lee, *Learning PySpark* (Packt, 2017, ISBN 9781786463708) — the
  beginner-friendly option, good if you want to get something running before you
  understand all of it.
- Chambers & Zaharia, *Spark: The Definitive Guide* (O'Reilly, 2018, ISBN
  9781491912218) — heavier, but Zaharia is one of the original Spark authors, so
  it's about as authoritative as it gets.

**Hadoop**
- Tom White, *Hadoop: The Definitive Guide*, 4th ed. (O'Reilly, 2015, ISBN
  9781491901687) — companion site at [hadoopbook.com](http://www.hadoopbook.com/).
  This is the standard reference even though it's a decade old at this point; the
  core MapReduce/HDFS mechanics haven't changed much.
- Chuck Lam, *Hadoop in Action* (Manning, 2010, ISBN 9781935182191) — older still,
  listed as an alternative.

Both books being 10–15 years old is a fair reflection of how settled this part of
the stack is. Spark evolves faster (hence the docs link to a specific version,
3.5.6, below) but the fundamentals of MapReduce and HDFS are basically frozen.


## 3. The Spark library ecosystem

Spark isn't one thing — it's a core engine plus four libraries built on top of it,
each covering a different kind of workload:

| Library | What it's for |
|---|---|
| **Spark SQL / DataFrames** | Parallel SQL-style querying, built for analytics rather than real-time transactional workloads |
| **MLlib** | Parallel machine learning |
| **GraphFrames** | Parallel graph processing (built *on top of* Spark, not part of core) |
| **Structured Streaming** | Processing data as it arrives, rather than in one big batch |

Documentation for the exact version used in the home assignments:
<https://spark.apache.org/docs/3.5.6/>

Commercial support is available from Databricks (founded by the original Spark
team) and Cloudera. Worth knowing this exists if you ever end up debugging a
production cluster and not just a laptop notebook.


## 4. RDDs — the original Spark abstraction

Before DataFrames existed, Spark worked entirely in terms of **RDDs** (Resilient
Distributed Datasets). You process an RDD with **transformations** (map, filter,
etc. — these are lazy, they just build up a plan) and **actions** (collect, count,
etc. — these actually trigger computation).

Full reference, matching the version used in this course's home assignments:
<https://spark.apache.org/docs/3.5.6/rdd-programming-guide.html>. It's worth
toggling the language selector on that page to Python, since the default examples
are often shown in Scala first.

### GraphFrames

GraphFrames sits on top of Spark (not part of core) and is aimed at large-scale
graph problems — the canonical example being social network analysis. You can
represent a graph as a `GraphFrame`, run simple structural queries, or implement
more complex algorithms using the **Pregel** bulk-synchronous-parallel (BSP)
computing model. It ships with common algorithms like triangle counting and
PageRank already implemented. GraphX is the older, RDD-based graph library for
Scala that GraphFrames effectively supersedes.

Docs: <https://docs.databricks.com/spark/latest/graph-analysis/graphframes/index.html>

### MLlib

MLlib is Spark's parallel machine learning library. It's not the widest collection
of algorithms out there, and it's not the fastest implementation of any single one
— scikit-learn or specialised libraries will often beat it on a single machine. Its
real selling point is tight integration with Spark SQL, so you can go from a
DataFrame straight into a training pipeline without shipping data somewhere else
first.

Docs: <https://spark.apache.org/docs/3.5.6/ml-guide.html>


## 5. Spark SQL and DataFrames

Spark SQL is a parallel SQL implementation tuned for analytics workloads, not
real-time transaction processing. DataFrames are the underlying data structure and
execution engine — you can use DataFrames directly without ever writing SQL, and
the two are interchangeable views onto the same execution plan.

The line from the slides that's worth remembering: *"SQL is the most widely used
parallel functional programming language."* It sounds like a throwaway joke, but
there's a real point underneath it — SQL queries describe *what* result you want,
not *how* to compute it step by step, which is exactly what lets the engine
parallelise and optimise freely. That's the same property functional programming
languages have, just with a much friendlier syntax for people who never studied
functional programming.

Docs: <http://spark.apache.org/docs/3.5.6/sql-programming-guide.html>

### Why DataFrames exist at all

RDDs are operated on by arbitrary user-defined functions in Scala, Java, or Python.
That flexibility is also the problem: if Spark has no idea what's inside your
lambda, it can't optimise around it. It just has to run exactly what you wrote.

DataFrames constrain you to a limited, known set of native operations. Because
Spark understands every operation in the pipeline, the SQL optimizer can rewrite
your DataFrame code into something equivalent but faster — reordering filters,
pushing predicates down, fusing operations — before anything actually runs. So
instead of literally executing what you wrote, Spark executes an optimized plan
that produces the same result, and can even compile parts of that plan rather than
interpreting it. This is the main reason DataFrames have mostly replaced RDDs for
everyday use, even though RDDs are still there underneath.


## 6. Broadcast variables

Broadcast variables send read-only data out to every worker in a coordinated way,
rather than shipping a copy along with every single task. Useful for things like a
lookup table that every task needs but that's too big to want serialized
repeatedly.


In [1]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

broadcastVar = sc.broadcast([1, 2, 3])
print(broadcastVar)        # <pyspark.broadcast.Broadcast object at ...>
print(broadcastVar.value)  # [1, 2, 3]


[1, 2, 3]


## 7. Accumulators

Accumulators let workers report statistics back to the driver during a job —
counters, sums, that kind of thing.


In [2]:
accum = sc.accumulator(0)
print(accum)  # Accumulator<id=0, value=0>

sc.parallelize([1, 2, 3, 4]).foreach(lambda x: accum.add(x))

print(accum.value)  # 10


0
10


**The catch:** if a transformation gets rescheduled during execution — say a
task fails and Spark retries it, or speculative execution kicks off a duplicate
task — the accumulator gets incremented more than once for the same logical piece
of work. So accumulators are fine for counting how much processing happened
(including "wasted" processing from retries), but they're not a safe tool for
doing an actual computation you depend on being exactly correct. If you need an
exact count of records processed, don't reach for an accumulator.


## 8. Closures — a classic footgun

When Spark actually runs in distributed mode (not just `local[*]` on your laptop),
any changes a worker makes to its own local variables disappear the moment that
worker's task finishes. The variable was copied into the closure that got shipped
to the worker; mutating it there does nothing to the original object back on the
driver.

The only real channel for a worker to communicate anything back is through the
data it writes into an RDD or DataFrame. This trips people up constantly — someone
writes something like a plain Python list that gets appended to inside a `map`,
runs it locally where it happens to "work" because everything's in one process,
then deploys it to a real cluster and can't figure out why the list is always
empty afterward. If you need shared state across workers, that state has to travel
through the RDD/DataFrame data path (or an accumulator, with the caveats above),
not through a captured local variable.


## 9. RDD / DataFrame persistence levels

By default, an RDD gets recomputed from scratch every time it's used in an action.
If you're going to reuse the same RDD or DataFrame multiple times, `persist()` (or
its shortcut `cache()`) tells Spark to keep it around instead. `unpersist()`
releases that memory again once you're done with it.

| Level | Where it's stored |
|---|---|
| `MEMORY_ONLY` | In memory as deserialized Java/Python objects |
| `MEMORY_AND_DISK` | Memory first, spills to disk if it doesn't fit |
| `MEMORY_ONLY_SER` *(Java/Scala)* | In memory, but serialized — more compact, costs CPU to deserialize |
| `MEMORY_AND_DISK_SER` *(Java/Scala)* | Serialized, spills to disk if needed |
| `DISK_ONLY` | Disk only |
| `MEMORY_ONLY_2`, `MEMORY_AND_DISK_ONLY_2` | Same as above but replicated on two nodes, for fault tolerance |
| `OFF_HEAP` *(experimental)* | Stored outside the JVM heap |

In practice `MEMORY_ONLY` and `MEMORY_AND_DISK` cover the vast majority of cases.
The `_SER` variants trade CPU time for memory footprint, and the `_2` variants
trade extra memory for not having to recompute from source if a node dies.


## 10. MapReduce, properly recapped

This is the part that explains *why* Spark is built the way it is — Spark's RDD
model grew directly out of the limitations of classic Hadoop MapReduce, so it's
worth going through the phases carefully.

**Phase 1 — Master startup.** A Master process (Hadoop calls this the Job Tracker)
coordinates the whole job. It's worth noting explicitly: the Master is a **single
point of failure**. If it dies mid-job, the job dies with it. This is one of the
big things Spark's driver/cluster-manager split and later Hadoop versions (YARN)
tried to address.

**Phase 2 — Worker allocation.** The Master creates *M* Map workers and assigns
each one an input split. Later it also starts *R* Reduce workers.

**Phase 3 — Map.** Input arrives at a free Map worker in chunks of roughly
64–128MB at a time. The user-supplied Map function is fed `(key, value)` pairs and
emits its own `(key, value)` pairs — the function itself has no idea it's running
on one machine out of thousands.

**Phase 4 — Local flush.** Map workers periodically write their output
`(key, value)` pairs to local disk, partitioned into *R* buckets by key (hashing
by default — one bucket per reduce worker).

**Phase 5 — Shuffle.** Once all input splits are processed, the shuffle phase
kicks off: up to *M × R* file transfers move mapper output across the network to
whichever reducer owns each key partition. Note that number — with 100 mappers and
100 reducers that's already 10,000 file transfers, which is exactly why shuffle is
usually the expensive part of a MapReduce or Spark job. Once a reducer has
received its input files, it sorts (and groups) the `(key, value)` pairs by key.

**Phase 6 — Reduce.** The user-supplied Reduce function iterates over
`(key, (…, list of values, …))` and emits final `(key, value)` output, one file per
reducer.

The framework guarantees all of this — the user only ever writes the Map and
Reduce functions. The *only* channel of communication between nodes is the
shuffle. Everything else (fault tolerance, parallelization, re-execution of failed
tasks) is handled automatically, and it's handled automatically *because* the
functional-programming style (pure functions, no side effects visible outside the
shuffle) makes re-running a failed task safe — you just run it again and get the
same result.

A nice side effect: because the framework controls partitioning, you can implement
a distributed sort just by supplying a custom partitioning function instead of the
default hash partitioner.


### Detailed Hadoop dataflow
![Detailed Hadoop dataflow](images/Hadoop_Dataflow.png)




### Combiners

A **combiner** is a Reduce function run early, on the Map side, before the
shuffle even happens. The idea: if you're about to sum a million values for the
same key, why ship all million of them across the network when you could sum them
locally first and ship one partial sum instead? This only works if the reduce
function is **associative and commutative** — order can't matter, and grouping
can't matter, because the combiner is only ever applying the reduce logic to
whatever subset happened to land on that particular mapper.

Spark actually *requires* this property and runs combiner-equivalent logic
automatically wherever it applies — you don't get a choice about it the way you do
in raw Hadoop MapReduce.


![Adding a combiner function](images/combiner.png)


## 11. Apache Hadoop

Hadoop is the open-source implementation of the MapReduce idea, originally built
by Doug Cutting and adopted heavily at Yahoo! and Facebook in its early years.

The design philosophy is summed up in one line worth memorising: **"Moving
Computation is Cheaper than Moving Data."** Ship the code to wherever the data
already lives, rather than pulling terabytes across the network to wherever the
code happens to be running. Concretely, this means Map and Reduce workers double
as storage nodes — job scheduling first tries to place work on a node that already
holds a copy of the relevant data, and falls back to a node in the same rack (to
keep network hops cheap) if that's not possible.

Reliability comes from replicating almost everything on commodity hardware that's
individually expected to fail — the two exceptions being the Master/Job Tracker
and the HDFS NameNode, both of which are historically single points of failure
(federation and later HA setups address this, more on that below).

Design targets, not incidental properties:
- Tuned for large files, gigabytes at a minimum
- Built for very large datasets — 1 PB+
- Optimized for streaming batch access, high bandwidth over low latency
- **Not** a POSIX filesystem — that constraint is deliberately traded away for
  scalability
- Written in Java, running as ordinary user-space daemons rather than kernel code

Project page: <http://hadoop.apache.org/>. HDFS specifically is still in very wide
use even where the original MapReduce execution engine has been replaced by Spark
or something else — the filesystem and the compute engine are separable, and HDFS
outlived classic MapReduce as the dominant piece.

When deciding whether MapReduce (or Spark, which inherits the same shape) is the
right fit for a problem, the thing to check is whether your algorithm can actually
be expressed in that fixed data-flow pattern — split, map, shuffle, reduce.
Algorithms that don't map cleanly onto that shape end up fighting the framework
instead of using it.


## 12. HDFS — the Hadoop Distributed Filesystem

**Core idea:** a distributed, replicated filesystem, inspired directly by the
original Google File System paper. Every block of data is replicated on three
different DataNodes by default, so any piece of data stays available as long as at
least one of its three replicas is up and reachable.

**Nodes:** each DataNode is typically an ordinary Linux machine with somewhere
between 8 and 24 hard disks. A single **NameNode** maintains the filesystem
metadata — which blocks live where — while potentially thousands of **DataNodes**
actually hold the data.

**Rack-awareness:** by default, one replica is written locally, a second replica
goes to another node in the same rack, and a third goes to a node in a *different*
rack. That third replica is specifically insurance against a whole-rack failure —
a rack switch dying, or a shared power feed tripping, which would otherwise take
out every replica at once if they were all in the same rack.

**Block size:** large by design, 128MB is a common default. This is a deliberate
trade against POSIX-style filesystems built for small files and low-latency random
access — HDFS is optimised for the opposite: big sequential batch reads.

**Write-once, read-many:** the big scalability trade-off. HDFS doesn't support
arbitrary in-place modification of files. This forces every system built on top of
it to only ever do sequential, append-style writes. That constraint sounds
restrictive, but it's what makes a huge amount of the Hadoop ecosystem work at
all — Spark and all of its libraries, HBase (a sequential-write database, roughly
a Bigtable clone), the MapReduce engine itself, Pig and Hive for data mining,
Mahout for machine learning, Lucene/Solr for full-text search, and Nutch for web
crawling all had to be designed around write-once semantics from the start.


### HDFS architecture

*("HDFS Under the Hood" diagram by Sanjay Radia of Yahoo — client, one
NameNode handling `getLocations`/`getFileInfo`/`create`/`addBlock`, and several
DataNodes exchanging blocks via `copy`, `replicate`, `blockReceived`, `read`, and
`write` operations. Original source, archived:
http://web.archive.org/web/20150701124307if_/http://assets.en.oreilly.com:80/1/event/12/HDFS%20Under%20the%20Hood%20Presentation%201.pdf)*


![HDFS architecture](images/hdfs.png)


### NameNode mechanics and scaling

The NameNode keeps all metadata **in memory** for speed, writes an edit log for
durability, and periodically snapshots that log to disk — a design that trades
NameNode RAM for lookup speed, which is also exactly why NameNode capacity became
the scalability bottleneck later on.

All actual data reads and writes go **directly** to the DataNodes — the NameNode
only ever hands out block locations, it never sits in the data path itself.
Replica writes are pipelined in a daisy-chain: the client writes to the first
DataNode, which streams the same data on to the second, which streams it on to the
third, rather than the client uploading three separate copies itself.

A newer feature, **HDFS Federation**, lets the namespace be split across multiple
NameNodes:
<https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/Federation.html>

### Scalability numbers

Some concrete figures, because "big" doesn't mean much without them:

- 20PB+ deployed HDFS installations (10,000+ hard disks) back in 2009; 100+ PB
  installations by 2018.
- 4,000+ DataNodes in large deployments.
- A **single** NameNode hits scalability limits on write-heavy workloads at around
  10,000 clients — this is documented in Shvachko's paper *"HDFS scalability: the
  limits to growth"*:
  <https://www.usenix.org/system/files/login/articles/1908-shvachko.pdf>.
  Federation was built specifically to address this scaling limit — note that it
  addresses *scalability*, not fault tolerance; those are separate problems with
  separate solutions.
- Other newer performance work includes reading from replica NameNodes
  (HDFS-12975), aimed at spreading out read load rather than write load.

Original design targets from around 2009, next to what had actually been deployed
by that point:

| Target | Original goal | Deployed (2009) | Deployed (2018) |
|---|---|---|---|
| Capacity | 10 PB | 14 PB | 100+ PB (Uber) |
| Nodes | 10,000 | 4,000 | — |
| Clients | 100,000 | 15,000 | — |
| Files | 100,000,000 | 60,000,000 | — |

Worth noticing: even the *original* 2009 targets had already been exceeded on
capacity by the time they were set as targets. Growth outran the roadmap.


## 13. Hardware and network considerations

**Typical node spec:** reasonable CPU, reasonable RAM, and 8–24 hard disks per
node. Because CPU speed has grown faster than hard disk throughput over the years,
newer deployments lean toward *more* disks per node rather than faster ones — you
get more aggregate I/O bandwidth that way. 10 Gigabit Ethernet is the current
dominant networking choice. Spark specifically leans on RAM for caching — 32GB per
node is treated as a sane minimum, with configurations up to 256GB being common
where the workload benefits from keeping large datasets resident in memory.

**Network bandwidth in practice:**
- Spark and Hadoop are both fairly insensitive to network *latency* — a slow round
  trip here and there doesn't hurt much, because the whole model is built around
  large sequential batch transfers rather than lots of small quick ones.
- Mapper reads can often come straight from local disk or, failing that, from
  elsewhere in the same rack — and intra-rack bandwidth is cheaper than
  inter-rack, so this is a meaningful cost saving, not just a nicety.
- Jobs with small Map output barely touch the network at all.
- Jobs with large Map output need serious inter-node bandwidth during the shuffle
  phase — this is where jobs actually get expensive.
- The practical takeaway: minimize the volume of shuffle data your job generates.
  This is the single lever with the biggest effect on network cost, which is why
  combiners (see above) exist in the first place — they exist specifically to
  shrink shuffle volume before the expensive part even starts.


## Summary

Spark and Hadoop MapReduce both start from the same constraint: computation is
cheap and plentiful, but moving data across a network is expensive, so the whole
architecture — HDFS placement, the shuffle phase, combiners, broadcast variables,
DataFrame optimization — is built around minimizing data movement rather than
minimizing raw compute. Once that constraint clicks, most of the individual design
choices in this lecture (rack-awareness, write-once semantics, the shift from RDDs
to optimizable DataFrames, why accumulators can double-count) stop looking like a
pile of unrelated facts and start looking like consequences of the same one idea.
